# CommGuard calibration v3

Canonical dual-T4 calibration source. It records environment and bounded collective observations; it makes no detector claim. Run with Internet enabled only for the immutable Git fetch, and never add credentials to this notebook.


In [ ]:
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_calibration_v3"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
REVIEWED_COMMIT = ""  # Required: immutable 40-character commit visible on origin.
REPOSITORY = Path("/kaggle/working/commguard-source")

if not re.fullmatch(r"[0-9a-f]{40}", REVIEWED_COMMIT):
    raise RuntimeError("Set REVIEWED_COMMIT to the reviewed, pushed 40-character commit SHA.")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin", REVIEWED_COMMIT], check=True)
subprocess.run(
    ["git", "-C", str(REPOSITORY), "checkout", "--detach", REVIEWED_COMMIT], check=True
)
head = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPOSITORY), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
pushed_refs = subprocess.run(
    ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if head != REVIEWED_COMMIT or dirty or not pushed_refs:
    raise RuntimeError(
        "Reproducibility gate failed: "
        f"head={head} dirty={bool(dirty)} pushed={bool(pushed_refs)}"
    )
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
        "-e", str(REPOSITORY),
    ],
    check=True,
)
print({"reviewed_commit": head, "remote_refs": pushed_refs.splitlines()})


In [ ]:
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")
if ARTIFACTS.exists():
    raise RuntimeError(f"Refusing to overwrite prior artifacts: {ARTIFACTS}")


In [ ]:
from datetime import datetime, timezone

from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CONTEXT = ProvenanceContext.create(
    corpus_id=f"corpus-calibration-v3-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-calibration-v3-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-calibration-v3-{NOTEBOOK_RUN_ID}",
    notebook_version="commguard_calibration_v3",
    input_archive_sha256=None,
    random_seed=20260730,
    repository_root=REPOSITORY,
)
if CONTEXT.source_dirty or CONTEXT.source_commit != REVIEWED_COMMIT:
    raise RuntimeError("SDK provenance no longer matches the clean reviewed source commit.")
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
from commguard.orchestrator import run_calibration_sweep

RUN_CALIBRATION_PILOT = True
RUN_EXPANDED_CALIBRATION = False
CALIBRATION = None
if RUN_CALIBRATION_PILOT:
    CALIBRATION = run_calibration_sweep(
        output=ARTIFACTS,
        payload_mib=(1, 16, 64),
        repetitions=1,
        timeout_s=180.0,
        provenance=CONTEXT,
    )
if RUN_EXPANDED_CALIBRATION:
    CALIBRATION = run_calibration_sweep(
        output=ARTIFACTS,
        payload_mib=(1, 4, 16, 64),
        repetitions=3,
        timeout_s=180.0,
        provenance=CONTEXT,
    )
if CALIBRATION is None:
    raise RuntimeError("Enable one calibration mode before export.")
print({"status": CALIBRATION["status"], "observations": len(CALIBRATION["observations"])})


## Results

not executed. The committed notebook contains no runtime result or output.


In [ ]:
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-calibration-v3-{NOTEBOOK_RUN_ID}.tar.gz")
ArtifactStore(ARTIFACTS).export(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
SHA_FILE = ARCHIVE.with_suffix(ARCHIVE.suffix + ".sha256")
SHA_FILE.write_text(f"{ARCHIVE_SHA256}  {ARCHIVE.name}\n", encoding="utf-8")
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in commguard_benign_corpus_v2.ipynb.")
print(
    f"NEXT STEP: set that notebook's REVIEWED_COMMIT to {REVIEWED_COMMIT} "
    "and run from the first cell."
)
